In [12]:
import csv
import os
from datetime import datetime

# ------ Restaurant Class ------ #

class Restaurant:
    def __init__(self):
        self.menu_file = "menu.csv"
        self.order_file = "orders.csv"
        self.menu = {}
        self.cart = []

        self.create_files()
        self.load_menu()

    # Create CSV files if not available

    def create_files(self):
        if not os.path.exists(self.menu_file):
            with open(self.menu_file, "w", newline="") as file:
                writer = csv.writer(file)
                writer.writerow(["ID", "Item", "Category", "Price"])
                writer.writerows([
                    [1, "Burger", "Fast Food", 120],
                    [2, "Pizza", "Fast Food", 250],
                    [3, "Pasta", "Italian", 180],
                    [4, "Cold Coffee", "Beverage", 90],
                    [5, "Paneer Tikka", "Starter", 220],
                    [6, "Biryani", "Main Course", 260]
                ])

        if not os.path.exists(self.order_file):
            with open(self.order_file, "w", newline="") as file:
                writer = csv.writer(file)
                writer.writerow(["Date", "Customer", "Items", "Total"])

    # Load menu into dictionary

    def load_menu(self):
        self.menu.clear()
        with open(self.menu_file, "r") as file:
            reader = csv.DictReader(file)
            for row in reader:
                self.menu[int(row["ID"])] = {
                    "item": row["Item"],
                    "category": row["Category"],
                    "price": int(row["Price"])
                }

    # Display menu

    def display_menu(self):
        print("\n" + "=" * 40)
        print("          RESTAURANT MENU")
        print("=" * 40)
        print(f"{'ID':<5}{'Item':<20}{'Price'}")
        print("-" * 40)
        for id, details in self.menu.items():
            print(f"{id:<5}{details['item']:<20}₹{details['price']}")
        print("=" * 40)

    # Add food to cart

    def add_to_cart(self):
        self.display_menu()
        try:
            food_id = int(input("\nEnter Food ID: "))
            if food_id not in self.menu:
                print("Invalid Food ID!")
                return

            qty = int(input("Enter Quantity: "))
            if qty <= 0:
                print("Quantity must be greater than 0.")
                return

            item = self.menu[food_id]
            self.cart.append({
                "id": food_id,
                "item": item["item"],
                "price": item["price"],
                "qty": qty
            })
            print(f"{item['item']} added successfully!")

        except ValueError:
            print("Please enter valid numbers.")

    # View Cart

    def view_cart(self):
        if not self.cart:
            print("\nCart is empty.")
            return

        print("\n" + "=" * 40)
        print("                YOUR CART")
        print("=" * 40)
        total = 0

        for i, item in enumerate(self.cart, 1):
            subtotal = item["price"] * item["qty"]
            total += subtotal
            print(f"{i}. {item['item']} x {item['qty']} = ₹{subtotal}")

        print("-" * 40)
        print(f"Total Amount: ₹{total}")
        print("=" * 40)

    # Remove Item

    def remove_item(self):
        self.view_cart()

        if not self.cart:
            return

        try:
            choice = int(input("Enter item number to remove: "))
            if 1 <= choice <= len(self.cart):
                removed = self.cart.pop(choice - 1)
                print(f"{removed['item']} removed.")
            else:
                print("Invalid choice.")
        except ValueError:
            print("Enter a valid number.")

    # Generate Bill

    def checkout(self):
        if not self.cart:
            print("Cart is empty.")
            return

        customer = input("Enter Customer Name: ")

        print("\n" + "=" * 40)
        print("             FINAL BILL")
        print("=" * 40)

        total = 0
        items = []

        for item in self.cart:
            subtotal = item["price"] * item["qty"]
            total += subtotal
            items.append(item["item"])
            print(f"{item['item']:<20}{item['qty']} x ₹{item['price']} = ₹{subtotal}")

        gst = total * 0.05
        final = total + gst

        print("-" * 55)
        print(f"Subtotal : ₹{total}")
        print(f"GST (5%) : ₹{gst:.2f}")
        print(f"Grand Total : ₹{final:.2f}")
        print("=" * 40)

        with open(self.order_file, "a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow([
                datetime.now().strftime("%d-%m-%Y %H:%M"),
                customer,
                ", ".join(items),
                round(final, 2)
            ])

        self.cart.clear()
        print("Order placed successfully!")

    # Order History

    def order_history(self):
        print("\n" + "=" * 40)
        print("              ORDER HISTORY")
        print("=" * 40)

        with open(self.order_file, "r") as file:
            reader = csv.reader(file)
            next(reader)

            found = False
            for row in reader:
                found = True
                print(f"\nDate     : {row[0]}")
                print(f"Customer : {row[1]}")
                print(f"Items    : {row[2]}")
                print(f"Total    : ₹{row[3]}")

            if not found:
                print("No previous orders found.")

    # Search Item

    def search_item(self):
        keyword = input("Enter food name: ").lower()

        found = False
        print("\nResults:")
        print("-" * 30)

        for id, item in self.menu.items():
            if keyword in item["item"].lower():
                found = True
                print(f"{id}. {item['item']} - ₹{item['price']}")

        if not found:
            print("No matching item found.")

    # Show Categories (Set & Tuple)

    def categories(self):
        category_set = {item["category"] for item in self.menu.values()}
        category_tuple = tuple(category_set)

        print("\nAvailable Categories:")
        for c in category_tuple:
            print("•", c)


# ------ Main Program ------ #

def main():
    restaurant = Restaurant()

    while True:
        print("\n" + "=" * 40)
        print("      RESTAURANT ORDERING SYSTEM")
        print("=" * 40)
        print("\n1. Display Menu")
        print("2. Add Food to Cart")
        print("3. View Cart")
        print("4. Remove Item")
        print("5. Search Food")
        print("6. View Categories")
        print("7. Checkout & Generate Bill")
        print("8. Order History")
        print("9. Exit\n")
        print("=" * 40)

        choice = input("\nEnter your choice: ")

        if choice == "1":
            restaurant.display_menu()

        elif choice == "2":
            restaurant.add_to_cart()

        elif choice == "3":
            restaurant.view_cart()

        elif choice == "4":
            restaurant.remove_item()

        elif choice == "5":
            restaurant.search_item()

        elif choice == "6":
            restaurant.categories()

        elif choice == "7":
            restaurant.checkout()

        elif choice == "8":
            restaurant.order_history()

        elif choice == "9":
            print("\nThank you for visiting our restaurant!")
            break

        else:
            print("Invalid choice! Please try again.")


if __name__ == "__main__":
    main()


      RESTAURANT ORDERING SYSTEM

1. Display Menu
2. Add Food to Cart
3. View Cart
4. Remove Item
5. Search Food
6. View Categories
7. Checkout & Generate Bill
8. Order History
9. Exit


Enter your choice: 2

          RESTAURANT MENU
ID   Item                Price
----------------------------------------
1    Burger              ₹120
2    Pizza               ₹250
3    Pasta               ₹180
4    Cold Coffee         ₹90
5    Paneer Tikka        ₹220
6    Biryani             ₹260

Enter Food ID: 3
Enter Quantity: 2
Pasta added successfully!

      RESTAURANT ORDERING SYSTEM

1. Display Menu
2. Add Food to Cart
3. View Cart
4. Remove Item
5. Search Food
6. View Categories
7. Checkout & Generate Bill
8. Order History
9. Exit


Enter your choice: 2

          RESTAURANT MENU
ID   Item                Price
----------------------------------------
1    Burger              ₹120
2    Pizza               ₹250
3    Pasta               ₹180
4    Cold Coffee         ₹90
5    Paneer Tikka       